# DataCenterGym Tutorial

Learn the basics of DataCenterGym in this interactive notebook.

In [ ]:
# Installation check
import sys
sys.path.insert(0, '..')

from datacenter_env import DataCenterEnv
from policies.all_policies import GreedyCapacity
import yaml, pickle, pandas as pd
import matplotlib.pyplot as plt
print("✓ All imports successful!")

## 1. Load Environment

In [ ]:
# Load configuration
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

# Load workload traces
with open('../bundles/bundle_p088_nominal/cluster_map.pkl', 'rb') as f:
    cluster_map = pickle.load(f)
job_trace = pd.read_parquet('../bundles/bundle_p088_nominal/job_trace.parquet')
usage_trace = pd.read_parquet('../bundles/bundle_p088_nominal/usage_trace.parquet')

print(f"Clusters: {len(cluster_map)}")
print(f"Jobs: {len(job_trace)}")

## 2. Run Simulation

In [ ]:
# Create environment and policy
env = DataCenterEnv(cluster_map, job_trace, usage_trace, None, config)
policy = GreedyCapacity(env)

# Run episode
obs, _ = env.reset()
temperatures = []
energies = []

for step in range(100):
    action = policy.select_action(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    temperatures.append(obs['datacenter_state'][0]['temperature'])
    energies.append(env.total_energy_kwh)
    if terminated or truncated:
        break

print(f"Steps: {len(temperatures)}")
print(f"Final Energy: {energies[-1]:.2f} kWh")

## 3. Visualize Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(temperatures)
ax1.set_xlabel('Timestep')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Datacenter Temperature')
ax1.grid(True)

ax2.plot(energies)
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Energy (kWh)')
ax2.set_title('Cumulative Energy Consumption')
ax2.grid(True)

plt.tight_layout()
plt.show()